In [16]:
import pandas as pd
import numpy as np

df=pd.read_csv("../data/processed/features.csv",parse_dates=["date"])
df=df[(df["date"]>="2006-01-03") & (df["date"]<="2019-12-31")]
train=df[df["date"]<"2017-01-01"]
validation=df[df["date"]>="2017-01-01"]
print("training set start date :",train["date"].min())
print("training set end date :",train["date"].max())
print("validation set start date :",validation["date"].min())
print("validation set end date :",validation["date"].max())


training set start date : 2006-01-03 00:00:00
training set end date : 2016-12-30 00:00:00
validation set start date : 2017-01-03 00:00:00
validation set end date : 2019-12-31 00:00:00


In [22]:
def qlike(y_true, y_pred):
    var_y_true=np.exp(2*y_true)
    var_y_pred=np.exp(2*y_pred)
    ratio_var=var_y_true/var_y_pred
    return np.mean(ratio_var-np.log(ratio_var)-1)

In [29]:
from sklearn.metrics import mean_squared_error, r2_score

y_valid_random_walk=validation['target_log_vol']
y_pred_random_walk=validation['log_vol_d']

print("Random Walk :")
print("MSE :",mean_squared_error(y_valid_random_walk,y_pred_random_walk))
print("R2 :",r2_score(y_valid_random_walk,y_pred_random_walk))
print("Qlike :",qlike(y_valid_random_walk,y_pred_random_walk))

Random Walk :
MSE : 0.19703579587274125
R2 : 0.07279479780556508
Qlike : 0.5263026917566765


In [30]:
from sklearn.linear_model import LinearRegression

har_features=["log_vol_d", "log_vol_w", "log_vol_m"]
X_train_har=train[har_features]
y_train_har=train['target_log_vol']
X_valid_har=validation[har_features]
y_valid_har=validation['target_log_vol']

In [35]:
lin_reg_har=LinearRegression()
lin_reg_har.fit(X_train_har,y_train_har)
y_pred_har=lin_reg_har.predict(X_valid_har)

print("Har - linear regression :")
print("MSE :",mean_squared_error(y_valid_har,y_pred_har))
print("R2 :",r2_score(y_valid_har,y_pred_har))
print("Qlike :",qlike(y_valid_har,y_pred_har))

print("linear regression coef :",lin_reg_har.coef_)
print("linear regression intercept :",lin_reg_har.intercept_)

Har - linear regression :
MSE : 0.13179831367949865
R2 : 0.37978740592388305
Qlike : 0.3389156342017161
linear regression coef : [0.21231742 0.3166093  0.40886055]
linear regression intercept : -0.3205626535229076


$$
\log RV_{t+1} = -0.321 + 0.212 \log RV_t + 0.317 \log RV_t^{(5)} + 0.409 \log RV_t^{(22)} + \varepsilon_{t+1}
$$

Where:
- $RV_{t+1}$: daily volatility on day $t+1$ (Garman-Klass estimator), the quantity we forecast.
- $RV_t$: daily volatility on day $t$.
- $RV_t^{(5)}$ and $RV_t^{(22)}$: average daily volatility over the last 5 and 22 trading days (one week and one month).
- $\varepsilon_{t+1}$: error term.

HAR clearly beats the random walk on all metrics (MSE 0.132 vs 0.197, QLIKE 0.339 vs 0.526). The monthly term has the largest coefficient (0.41), then the weekly (0.32) and the daily term (0.21): the average level of volatility over the last month is more informative than yesterday's volatility alone, which is noisy. This confirms the long memory of volatility seen in the autocorrelation plot.